# Wine Quality EDA: What Makes a Good Wine?

**Dataset:** UCI Machine Learning Repository - Wine Quality  
**Source:** https://archive.ics.uci.edu/dataset/186/wine+quality

---

## Research Questions

1. **Which physicochemical properties are most strongly associated with higher wine quality?**
2. **Do red and white wines follow the same patterns, or do different chemical factors matter for each type?**

The notebook is organized into 5 sections: data loading, distribution overview, correlation analysis, high vs low quality comparison, and a type-by-type breakdown.

## Section 0: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Section 1: Load and Prepare the Data

In [ ]:
red = pd.read_csv('https://raw.githubusercontent.com/RickyRBs/Python_Midterm/main/winequality-red.csv', sep=';')
white = pd.read_csv('https://raw.githubusercontent.com/RickyRBs/Python_Midterm/main/winequality-white.csv', sep=';')

In [ ]:
red['type'] = 'red'
white['type'] = 'white'

In [ ]:
wine = pd.concat([red, white], ignore_index=True)

wine.columns = wine.columns.str.replace(' ', '_')

In [ ]:
wine['type'] = wine['type'].astype('category')

In [ ]:
print(f'Shape: {wine.shape}')
print(f'Missing values: {wine.isnull().sum().sum()}')
print(f'Wine types: {wine["type"].value_counts().to_dict()}')

In [ ]:
wine.head()

In [ ]:
wine.dtypes

In [ ]:
wine.describe().round(2)

## Section 2: Quality Distribution and Feature Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'red': '#C0392B', 'white': '#2980B9'}

for ax, wtype in zip(axes, ['red', 'white']):
    subset = wine.loc[wine['type'] == wtype]
    quality_counts = subset['quality'].value_counts().sort_index()
    ax.bar(quality_counts.index, quality_counts.values,
           color=colors[wtype], edgecolor='white', alpha=0.85, width=0.6)
    ax.set_title(f'{wtype.capitalize()} Wine (n={len(subset):,})')
    ax.set_xlabel('Quality Score')
    ax.set_ylabel('Count')
    ax.set_xticks(range(3, 10))

plt.suptitle('Distribution of Wine Quality Scores by Type', fontweight='bold')
plt.tight_layout()
plt.show()

Both types are concentrated at scores 5 and 6. White wine has a slightly wider spread and more high-scoring examples. Since the middle scores dominate, I defined high quality as ≥7 and low quality as ≤5 for the comparisons later on.

In [ ]:
numeric_features = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
    'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide',
    'density', 'pH', 'sulphates', 'alcohol'
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(numeric_features):
    ax = axes[i]
    for wtype, color in colors.items():
        subset = wine.loc[wine['type'] == wtype][feature]
        ax.hist(subset, bins=30, alpha=0.5, color=color, label=wtype, density=True)
    ax.set_title(feature.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

axes[-1].set_visible(False)

plt.suptitle('Chemical Feature Distributions: Red vs White Wine', fontweight='bold')
plt.tight_layout()
plt.show()

A few features look very different between red and white wine. White wine has much higher residual sugar and total sulfur dioxide, while red wine has higher volatile acidity. Several features are also heavily right skewed, which is why log transformations are applied before computing correlations in the next section.

## Section 3: Which Features Correlate with Quality?

In [ ]:
skewed = ['residual_sugar', 'chlorides', 'free_sulfur_dioxide',
          'total_sulfur_dioxide', 'sulphates', 'volatile_acidity', 'citric_acid']

wine_tf = wine.copy()
for col in skewed:
    wine_tf[col] = np.log(wine_tf[col] + 0.1)

print('Log transformation applied to:', skewed)

In [ ]:
corr_red = []
corr_white = []

for feature in numeric_features:
    red_subset = wine_tf.loc[wine_tf['type'] == 'red']
    white_subset = wine_tf.loc[wine_tf['type'] == 'white']
    
    r_red = np.corrcoef(red_subset[feature], red_subset['quality'])[0, 1]
    r_white = np.corrcoef(white_subset[feature], white_subset['quality'])[0, 1]
    
    corr_red.append(round(r_red, 3))
    corr_white.append(round(r_white, 3))

corr_df = pd.DataFrame({
    'feature': numeric_features,
    'red': corr_red,
    'white': corr_white
})

corr_df['abs_red'] = corr_df['red'].abs()
corr_df = corr_df.sort_values('abs_red', ascending=False).drop(columns='abs_red')
corr_df

In [ ]:
x = np.arange(len(corr_df))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(x + width/2, corr_df['red'].values,   width, label='Red',   color='#C0392B', alpha=0.85)
ax.barh(x - width/2, corr_df['white'].values, width, label='White', color='#2980B9', alpha=0.85)

ax.set_yticks(x)
ax.set_yticklabels([f.replace('_', ' ').title() for f in corr_df['feature']])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Quality')
ax.set_title('Feature Correlations with Wine Quality: Red vs White', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

Alcohol is the strongest positive predictor for both types (r ≈ 0.48 red, r ≈ 0.44 white). Some features tell pretty different stories though. Sulphates and citric acid are clearly positive for red wine but near zero for white, and density is a stronger negative predictor for white wine. This already hints that one model for both types probably misses something.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, wtype in zip(axes, ['red', 'white']):
    subset = wine_tf.loc[wine_tf['type'] == wtype][numeric_features + ['quality']]
    corr_matrix = subset.corr().round(2)
    
    #only show lower triangle
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, ax=ax, mask=mask,
                cmap='RdBu_r', vmin=-1, vmax=1,
                annot=True, fmt='.2f', annot_kws={'size': 7},
                linewidths=0.4, square=True)
    ax.set_title(f'{wtype.capitalize()} Wine', fontweight='bold')

plt.suptitle('Pairwise Feature Correlations by Wine Type', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

The heatmaps show some notable collinearity. Density is closely linked to residual sugar (r ≈ 0.84 in white wine) and inversely linked to alcohol. Free and total sulfur dioxide are also strongly correlated with each other, especially in white wine.

## Section 4: High-Quality vs Low-Quality Wines

In [ ]:
low = wine.loc[wine['quality'] <= 5].copy()
high = wine.loc[wine['quality'] >= 7].copy()

low['quality_group'] = 'low (<=5)'
high['quality_group'] = 'high (>=7)'

low_high = pd.concat([low, high], ignore_index=True)

print(f'Low quality wines: {len(low)}')
print(f'High quality wines: {len(high)}')

In [ ]:
key_features = ['alcohol', 'volatile_acidity', 'sulphates', 'density']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

quality_colors = {'low (<=5)': '#E67E22', 'high (>=7)': '#27AE60'}

for ax, feature in zip(axes, key_features):
    sns.violinplot(data=low_high, x='quality_group', y=feature,
                   hue='quality_group',
                   order=['low (<=5)', 'high (>=7)'],
                   palette=quality_colors, inner='box', cut=0,
                   legend=False, ax=ax)
    ax.set_title(feature.replace('_', ' ').title())
    ax.set_xlabel('Quality Group')
    ax.set_ylabel(feature.replace('_', ' '))

plt.suptitle('Key Feature Distributions: High vs Low Quality Wines', fontweight='bold')
plt.tight_layout()
plt.show()

The biggest gap between high and low quality wines is in alcohol content. High quality wines average around 11.5% ABV vs 9.8% for low quality. Volatile acidity is clearly lower in high quality wines. Density also drops in high quality wines, which makes sense since higher alcohol means lower density.

In [ ]:
focus_features = ['alcohol', 'volatile_acidity', 'citric_acid', 'sulphates']

results = []
for wtype in ['red', 'white']:
    for group in ['low (<=5)', 'high (>=7)']:
        subset = low_high.loc[(low_high['type'] == wtype) & (low_high['quality_group'] == group)]
        row = {'type': wtype, 'group': group}
        for f in focus_features:
            row[f] = round(subset[f].mean(), 3)
        results.append(row)

means_df = pd.DataFrame(results)
means_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, feature in enumerate(focus_features):
    ax = axes[i]
    sns.barplot(data=means_df, x='group', y=feature, hue='type',
                order=['low (<=5)', 'high (>=7)'],
                palette={'red': '#C0392B', 'white': '#2980B9'},
                ax=ax, alpha=0.85)
    ax.set_title(feature.replace('_', ' ').title())
    ax.set_xlabel('Quality Group')
    ax.set_ylabel('Mean Value')
    ax.legend(title='Type')

plt.suptitle('Mean Feature Values: Low vs High Quality by Wine Type', fontweight='bold')
plt.tight_layout()
plt.show()

Breaking it down by type shows something interesting. Sulphates have a clear gap between high and low quality red wines, but almost no difference for white wines. Volatile acidity drops in high quality wines for both types, but the drop is bigger for red. Alcohol is consistently higher in high quality wines no matter the type.

## Section 5: Do Red and White Wines Follow the Same Rules?

In [ ]:
def scatter_regression(ax, feature, title):
    """plot quality vs a feature with separate regression lines for red and white"""
    for wtype, color in [('red', '#C0392B'), ('white', '#2980B9')]:
        subset = wine.loc[wine['type'] == wtype]
        x = subset[feature].values
        y = subset['quality'].values
        
        ax.scatter(x, y, alpha=0.07, color=color, s=15)
        
        coeffs = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = coeffs[0] * x_line + coeffs[1]
        
        r = round(np.corrcoef(x, y)[0, 1], 2)
        ax.plot(x_line, y_line, color=color, linewidth=2,
                label=f'{wtype.capitalize()} (r={r})')
    
    ax.set_xlabel(feature.replace('_', ' ').title())
    ax.set_ylabel('Quality Score')
    ax.set_title(title)
    ax.legend(fontsize=10)
    ax.set_yticks(range(3, 10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter_regression(axes[0], 'alcohol', 'Alcohol vs Quality')
scatter_regression(axes[1], 'volatile_acidity', 'Volatile Acidity vs Quality')

plt.suptitle('Quality vs Key Features: Regression Lines by Wine Type', fontweight='bold')
plt.tight_layout()
plt.show()

For alcohol, the regression lines for red and white run nearly parallel, so alcohol predicts quality pretty consistently across both types. For volatile acidity, both lines slope downward but the red wine line is steeper. That means high volatile acidity seems to hurt red wine quality more than white.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter_regression(axes[0], 'sulphates', 'Sulphates vs Quality')
scatter_regression(axes[1], 'citric_acid', 'Citric Acid vs Quality')

plt.suptitle('Divergent Predictors: Features Where Red and White Differ', fontweight='bold')
plt.tight_layout()
plt.show()

Sulphates and citric acid are where red and white diverge most clearly. Both features are positively associated with red wine quality, but essentially flat for white wine.

In [ ]:
bins = [(8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 15)]
bin_labels = ['8-9%', '9-10%', '10-11%', '11-12%', '12-13%', '13-15%']

bin_results = []
for (lo, hi), label in zip(bins, bin_labels):
    for wtype in ['red', 'white']:
        subset = wine.loc[
            (wine['alcohol'] >= lo) & (wine['alcohol'] < hi) & (wine['type'] == wtype)
        ]
        if len(subset) > 0:
            bin_results.append({
                'alcohol_bin': label,
                'type': wtype,
                'mean_quality': round(subset['quality'].mean(), 2),
                'count': len(subset)
            })

bin_df = pd.DataFrame(bin_results)
bin_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for wtype, color in [('red', '#C0392B'), ('white', '#2980B9')]:
    subset = bin_df.loc[bin_df['type'] == wtype]
    ax.plot(subset['alcohol_bin'], subset['mean_quality'],
            marker='o', markersize=8, color=color, linewidth=2, label=wtype.capitalize())

ax.set_xlabel('Alcohol Content (% ABV)')
ax.set_ylabel('Mean Quality Score')
ax.set_title('Mean Quality Score by Alcohol Bin and Wine Type', fontweight='bold')
ax.legend()
ax.set_ylim(4.5, 7.5)
plt.tight_layout()
plt.show()

Mean quality goes up steadily with alcohol content for both types, and the trend lines are close to parallel. At similar alcohol levels, white wine tends to score slightly higher on average.

In [ ]:
corr_df['diff'] = (corr_df['red'] - corr_df['white']).abs()
corr_sorted = corr_df.sort_values('diff', ascending=False).set_index('feature')[['red', 'white']]
corr_sorted.index = [f.replace('_', ' ').title() for f in corr_sorted.index]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_sorted, ax=ax, cmap='RdBu_r', vmin=-0.5, vmax=0.5,
            annot=True, fmt='.2f', annot_kws={'size': 11},
            linewidths=0.5, linecolor='white')
ax.set_title('Feature-Quality Correlations by Wine Type\n(sorted by divergence)', fontweight='bold')
ax.set_xlabel('Wine Type')
plt.tight_layout()
plt.show()

The features at the top of the table (sulphates, citric acid, volatile acidity) are where red and white diverge most. Alcohol sits near the bottom, confirming it as the most consistent predictor across both wine types.

## Section 6: Conclusions

### Question 1: Which features are most associated with wine quality?

**Alcohol** is the single strongest predictor (r ≈ 0.46 overall). Wines above 13% ABV score nearly a full point higher than wines below 9%. **Volatile acidity** is the strongest negative predictor (r ≈ -0.27), which makes sense since it is linked to a vinegar-like taste.

### Question 2: Do red and white wines follow the same rules?

Partly yes, partly no. Alcohol matters equally for both types. But sulphates and citric acid are positive predictors only for red wine, while density and chlorides are stronger negative predictors for white wine.

| Feature | Red r | White r | Note |
|---|---|---|---|
| Alcohol | +0.48 | +0.44 | Consistent |
| Volatile acidity | -0.39 | -0.19 | Steeper for red |
| Sulphates | +0.25 | +0.05 | Red only |
| Citric acid | +0.23 | -0.01 | Red only |
| Density | -0.17 | -0.31 | Stronger for white |
| Chlorides | -0.13 | -0.21 | Stronger for white |